# Math Competition Further Investigation

The investigation compares reasoning-focused local models on the math game, with emphasis on timing behavior, optional priming, and whether Qwen benefits from a calculator tool.


### Imports and Runtime Dependencies

This cell gathers the libraries needed for the math investigation: standard utilities, tabular result handling, GPU/model loading, symbolic math tools, optimization helpers, and Hugging Face generation controls.


In [1]:
import importlib.util
import itertools
import gc
import json
import math
import os
import random
import re
import sys
import time
from collections import defaultdict
from dataclasses import dataclass
from statistics import mean, median, mode, stdev, variance

import pandas as pd
import torch
from scipy.optimize import linprog, minimize, minimize_scalar
from sympy import (
    E,
    I,
    Matrix,
    Rational,
    binomial,
    cos,
    diff,
    exp,
    expand,
    factor,
    factorial,
    factorint,
    gcd,
    integrate,
    isprime,
    lcm,
    limit,
    log,
    oo,
    pi,
    simplify,
    sin,
    solve,
    sqrt,
    symbols,
    tan,
)
import sympy
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    StoppingCriteria,
    StoppingCriteriaList,
)

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

### Colab Drive and Package Access

These setup cells mount Google Drive and add project folders to the Python path so the notebook can import the `millionaire_client` package and any local helper files used by the game wrapper.


In [2]:
from google.colab import drive

drive.mount('/content/gdrive')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [3]:
import sys
import os

# Define the path to the directory containing your package
package_parent_dir = '/content/gdrive/MyDrive/NLP_Project'

# Append to sys.path if it is not already present
if package_parent_dir not in sys.path:
    sys.path.append(package_parent_dir)

# Verify the path was added
print(sys.path)

['/content', '/env/python', '/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload', '', '/usr/local/lib/python3.12/dist-packages', '/usr/lib/python3/dist-packages', '/usr/local/lib/python3.12/dist-packages/IPython/extensions', '/root/.ipython', '/content/gdrive/MyDrive/NLP_Project']


In [4]:
import os, sys

package_parent_dir = "/content/gdrive/MyDrive/NLP_Project"
package_dir = package_parent_dir + "/millionaire_client"

print("Parent exists:", os.path.exists(package_parent_dir))
print("Package exists:", os.path.exists(package_dir))
print("__init__ exists:", os.path.exists(package_dir + "/__init__.py"))
print("Files:", os.listdir(package_dir) if os.path.exists(package_dir) else "missing")

if package_parent_dir not in sys.path:
    sys.path.append(package_parent_dir)

print("Path added:", package_parent_dir in sys.path)

Parent exists: True
Package exists: True
__init__ exists: True
Files: ['base.py', '__init__.py', 'leaderboard.py', 'exceptions.py', 'auth.py', 'game.py', 'competitions.py', 'client.py', 'models.py', '__pycache__']
Path added: True


In [5]:
import importlib.util
import sys
import os

package_parent_dir = "/content/gdrive/MyDrive/NLP_Project"
package_dir = os.path.join(package_parent_dir, "millionaire_client")

# Make sure parent is visible
if package_parent_dir not in sys.path:
    sys.path.insert(0, package_parent_dir)

# Load package manually
init_file = os.path.join(package_dir, "__init__.py")

spec = importlib.util.spec_from_file_location(
    "millionaire_client",
    init_file,
    submodule_search_locations=[package_dir]
)

millionaire_client = importlib.util.module_from_spec(spec)
sys.modules["millionaire_client"] = millionaire_client
spec.loader.exec_module(millionaire_client)

from millionaire_client import MillionaireClient, AuthenticationError

print("Import successful")

Import successful


### Login and Competition Selection

This block authenticates with the PoliMillionaire client, lists the available competitions for confirmation, and selects competition `3`, the math competition, as the target experiment.


In [6]:
from millionaire_client import MillionaireClient, AuthenticationError

API_URL = "http://131.175.15.22:51111/"
username = "amirparsa"
password = "@Mirparsa96"

client = MillionaireClient(API_URL)
try:
    user = client.login(username, password)
    print(f"\nWelcome, {user.username}! (Role: {user.role})")
except AuthenticationError as e:
    print(f"Login failed: {e}")


Welcome, amirparsa! (Role: student)


In [7]:
# List available competitions
print("\n=== Available Competitions ===")
competitions = client.competitions.list_all()
for comp in competitions:
    print(f"  {comp.id}: {comp.name} ({comp.max_levels} questions)")


=== Available Competitions ===
  0: Entertainment (15 questions)
  1: Ancient History and Politics (15 questions)
  2: Science and Nature (15 questions)
  3: Maths (15 questions)
  4: Philosophy and Psychology (15 questions)
  5: News (15 questions)


In [8]:
comp_id = 3

### Manual Game API Check

These cells provide a small manual-play path for sanity checking the game API before running automatic experiments. They start a game, allow answers to be submitted interactively, and inspect the leaderboard so the client connection can be verified.


In [9]:
def play_game(game):
  # Play the game
  while game.in_progress:
      question = game.current_question
      if not question:
          print("No question available. Game may have ended.")
          break

      print(f"\n--- Level {game.current_level} ---")
      print(f"Q: {question.text}")
      print()

      for opt in question.options:
          print(f"  [{opt.id}] {opt.text}")

      # Get time remaining
      time_left = game.time_remaining
      if time_left:
          print(f"\nTime remaining: {time_left:.1f}s")

      # Get answer
      try:
          answer_input = input("\nYour answer (option ID): ").strip()
          answer_id = int(answer_input)
      except ValueError:
          print("Invalid input. Please enter a number.")
          continue

      # Submit answer
      result = game.answer(answer_id)

      if result.correct:
          print(" CORRECT!")
          if result.game_over:
              print(f"\n CONGRATULATIONS! You completed the game!")
              print(f" Final earnings: ${result.earned_amount:,.2f}")
          else:
              print(f" Earned so far: ${result.earned_amount:,.2f}")
      elif result.timed_out:
        print("TIMED OUT!")
        print(f"\n Game Over!")
        print(f" Final earnings: ${result.earned_amount:,.2f}")
      elif not result.correct:
          print(" WRONG ANSWER!")
          print(f"\n Game Over!")
          print(f" Final earnings: ${result.earned_amount:,.2f}")

  print("\n=== Game Summary ===")
  print(f"Reached Level: {game.current_level}")
  print(f"Total Earnings: ${game.earned_amount:,.2f}")

In [10]:
# Start the game
print("\n=== Starting Game ===")
game = client.game.start(competition_id=comp_id)
print(f"Session ID: {game.session_id}")
print(f"Total number of questions: {game.state.competition.max_levels}")
print()
play_game(game)


=== Starting Game ===
Session ID: 312774
Total number of questions: 15


--- Level 1 ---
Q: A dentist has noticed that about two children in every seven whom he sees professionally develop cavities before they turn 10 years old. Last week he examined the teeth of five unrelated children younger than 10. Let X be the number of children who develop cavities before turning 10. Which of the following gives the probability that at least one will develop a cavity before turning 10?

  [0] 1 – P(X = 0)
  [1] P(X = 2 out of 7)
  [2] P(X = 2, 3, 4, 5, 6, 7)
  [3] P(X = 1)

Time remaining: 29.9s

Your answer (option ID): 3
TIMED OUT!

 Game Over!
 Final earnings: $0.00

=== Game Summary ===
Reached Level: 1
Total Earnings: $0.00


In [11]:
# Show leaderboard position
lb = client.leaderboard.get(competition_id=comp_id, limit=10)
print(f"\n=== Leaderboard for {lb.competition.name} ===")
for i, entry in enumerate(lb.entries[:5], 1):
    marker = " <-- YOU" if entry.username == username else ""
    print(f"  {i}. {entry.username}: ${entry.score:,.2f} (Level {entry.reached_level}){marker}")


=== Leaderboard for Maths ===
  1. leonfuss: $1,024,000.00 (Level 15)
  2. El_Marzel: $1,024,000.00 (Level 15)
  3. cemalgndzz: $1,024,000.00 (Level 15)
  4. Khashayar: $1,024,000.00 (Level 15)
  5. leonardo_pascotto: $1,024,000.00 (Level 15)


# Automatic Question Solver

The rest of the notebook builds the automatic solver stack used for the math experiments: prompt formatting, logging, model-output parsing, local model loading, timed reasoning, and ablation execution.


### Question Formatting Helper

`format_question` converts the game question object into a compact prompt with the question text, numbered answer options, and an explicit answer area. All solvers reuse this formatting so each model receives a consistent multiple-choice prompt.


In [12]:
def format_question(question):
    """
    Strong prompt forcing structured output.
    """

    options_text = "\n".join(
        [f"{opt.id}. {opt.text}" for opt in question.options]
    )

    return f"""
          Question:
          {question.text}

          Options:
          {options_text}

          Answer:
          """.strip()

### Optional Manual Auto-Game Start

This cell starts a game outside the ablation runner. It is useful for quick local checks, but the main experiment loop below starts its own games for each configured run.


In [13]:
# Start the game
print("\n=== Starting Game ===")
game = client.game.start(competition_id=comp_id)
question = game.current_question

print(format_question(question))


=== Starting Game ===
Question:
          Statement 1 | If H is a subgroup of a group G and a belongs to G, then aH = Ha. Statement 2 | If H is normal of G and a belongs to G, then ah = ha for all h in H.

          Options:
          0. True, True
1. True, False
2. False, True
3. False, False

          Answer:


### Random Solver Baseline

The random solver is a simple control strategy. It is not meant to be competitive; it gives a sanity baseline for the logging and answer-submission pipeline before local LLM solvers are used.


In [14]:
import random

def random_solver(question):
    return random.choice([opt.id for opt in question.options]), ""

### Game Logging and Automatic Play Loop

`run_logs` stores one row per answered question, while `game_logs` stores one row per completed game.

`log_question_result` records selected answer, correctness, timeout status, earned amount, decision time, and model reasoning text. `play_game_auto` runs a full game with a supplied solver, submits each answer, logs the outcome, and appends a per-game summary.


In [15]:
run_logs = []
game_logs = []

In [16]:
def log_question_result(
    run_id,
    solver_name,
    model_name,
    level,
    question,
    selected_answer,
    result,
    time_taken,
    thinking="",
):
    run_logs.append(
        {
            "run_id": run_id,
            "solver": solver_name,
            "model": model_name,
            "level": level,
            "question": question.text,
            "options": {opt.id: opt.text for opt in question.options},
            "selected_answer": selected_answer,
            "correct": result.correct,
            "timed_out": result.timed_out,
            "earned_amount": result.earned_amount,
            "decision_time_sec": round(time_taken, 3),
            "thinking": thinking,
        }
    )

In [17]:
def play_game_auto(game, solver_function, solver_name, model_name, run_id):
    game_start_log_idx = len(run_logs)

    while game.in_progress:
        question = game.current_question
        if not question:
            print("No question available.")
            break

        current_level = game.current_level
        print(f"\n--- Level {current_level} ---")
        print(format_question(question))

        time_left = game.time_remaining
        if time_left:
            print(f"\nTime remaining: {time_left:.1f}s")

        start_time = time.time()
        answer_id, thinking = solver_function(question)
        time_taken = time.time() - start_time

        print(f"\nSelected answer: {answer_id}")
        print(f"Decision time: {time_taken:.2f}s")

        result = game.answer(answer_id)

        log_question_result(
            run_id=run_id,
            solver_name=solver_name,
            model_name=model_name,
            level=current_level,
            question=question,
            selected_answer=answer_id,
            result=result,
            time_taken=time_taken,
            thinking=thinking,
        )

        if result.correct:
            print("CORRECT!")
        elif result.timed_out:
            print("TIMED OUT!")
        else:
            print("WRONG ANSWER!")

    game_question_logs = run_logs[game_start_log_idx:]
    answered_levels = [row["level"] for row in game_question_logs]
    decision_times = [row["decision_time_sec"] for row in game_question_logs]
    timeout_count = sum(1 for row in game_question_logs if row["timed_out"])

    summary = {
        "run_id": run_id,
        "solver": solver_name,
        "model": model_name,
        "levels_reached": game.current_level,
        "decision_time": mean(decision_times) if decision_times else None,
        "total_decision_time": sum(decision_times),
        "questions_answered": len(answered_levels),
        "timeouts": timeout_count,
        "earned_amount": game.earned_amount,
    }
    game_logs.append(summary)

    print("\n=== Game Summary ===")
    print(f"Reached Level: {game.current_level}")
    print(f"Total Earnings: ${game.earned_amount:,.2f}")
    return summary

### Inspect Raw Question Logs

This quick DataFrame view is useful during development to confirm that per-question logging is being populated correctly before running a full ablation batch.


In [18]:
import pandas as pd

df_logs = pd.DataFrame(run_logs)
df_logs

""


### Model Output Cleaning and Answer Parsing

These helpers turn raw model text into a valid option ID. They remove thinking/tool markup, look for valid numeric option IDs, and fall back to matching option text when the model answers with words instead of an index.


In [19]:
def clean_model_response(response):
    response = re.sub(r"<think>.*?</think>", "", response, flags=re.DOTALL)
    response = re.sub(r"<tool_call>.*?</tool_call>", "", response, flags=re.DOTALL)
    response = re.sub(r"<tool_response>.*?</tool_response>", "", response, flags=re.DOTALL)
    return response.replace("<|im_end|>", "").replace("<|im_start|>", "").strip()

In [20]:
def parse_model_answer(response, question):
    """
    Parse model output into a valid option ID.
    Handles numeric answers and option-text answers.
    """
    response = re.sub(r"<think>.*?</think>", "", response, flags=re.DOTALL)
    response = re.sub(r"<tool_call>.*?</tool_call>", "", response, flags=re.DOTALL)
    response = re.sub(r"<tool_response>.*?</tool_response>", "", response, flags=re.DOTALL)
    response = response.replace("<|im_end|>", "").replace("<|im_start|>", "").strip()

    response_clean = response.lower()
    valid_option_ids = [opt.id for opt in question.options]

    for match in reversed(list(re.finditer(r"\b(\d+)\b", response_clean))):
        num = int(match.group(1))
        if num in valid_option_ids:
            return num

    for opt in question.options:
        option_text = opt.text.strip().lower()
        if option_text and (option_text in response_clean or response_clean in option_text):
            return opt.id

    return None

In [21]:
def parse_final_answer(response, question):
    """
    Parse a short completion after prompts like "The answer index is".
    Searches from the beginning because the first digit should be the answer;
    later digits are often explanation steps.
    """
    response = clean_model_response(response)
    response_clean = response.lower()
    valid_option_ids = [opt.id for opt in question.options]

    for match in re.finditer(r"\b(\d+)\b", response_clean):
        num = int(match.group(1))
        if num in valid_option_ids:
            return num

    for opt in question.options:
        option_text = opt.text.strip().lower()
        if option_text and (option_text in response_clean or response_clean in option_text):
            return opt.id

    return None

# Local LLM Solver

This section defines the local-model experiment machinery. The design is the same as the commented math section in the NLP notebook: test DeepSeek and Qwen under controlled time budgets, compare soft-deadline behavior, and isolate whether Qwen improves when calculator tool calls are enabled.


### Model Runtime Dependencies

This install cell pins the core Hugging Face runtime packages used by the local LLM experiments.


In [22]:
!pip install -q --upgrade transformers==4.47.0 accelerate sentencepiece bitsandbytes

### Math Investigation Configuration

This block defines model checkpoints and the experiment configuration object. Each configuration controls model family, priming, soft and hard deadline behavior, calculator-tool access, warning timings, and number of games.

The validation in `ExperimentConfig` keeps invalid comparisons out of the experiment, such as enabling the Qwen-only calculator tool for DeepSeek or setting a soft deadline after the hard deadline.


In [23]:
# =============================================================================
# Model and solver configuration
# =============================================================================

QWEN_MODEL = "Qwen/Qwen2.5-7B-Instruct"
DEEPSEEK_MODEL = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"


@dataclass
class ExperimentConfig:
    model_name: str
    model_family: str

    use_priming: bool = True
    use_hard_deadline: bool = True
    use_soft_deadline: bool = True
    use_calc_tool: bool = False

    # Warning semantics:
    # - warnings 0 through n-2 inject a speed-up prompt and continue generation.
    # - warning n-1 is the soft deadline and injects a wrap-up prompt.
    # - t_hard_s remains a separate hard deadline after the soft deadline.
    warning_times_s: tuple = (22.0,)
    t_soft_s: float = 22.0
    t_hard_s: float = 27.0
    t_answer_s: float = 29.0
    max_tool_calls: int = 8
    num_games: int = 10

    def __post_init__(self):
        self.model_family = self.model_family.lower()
        if self.model_family not in {"qwen", "deepseek"}:
            raise ValueError("model_family must be 'qwen' or 'deepseek'")
        if self.use_calc_tool and self.model_family == "deepseek":
            raise ValueError("DeepSeek-R1 cannot use the math tool. Set use_calc_tool=False.")
        if self.use_soft_deadline and not self.use_hard_deadline:
            raise ValueError("use_soft_deadline requires use_hard_deadline=True")
        if self.use_soft_deadline:
            if not self.warning_times_s:
                self.warning_times_s = (self.t_soft_s,)
            self.warning_times_s = tuple(sorted(float(t) for t in self.warning_times_s))
            if self.warning_times_s[-1] >= self.t_hard_s:
                raise ValueError("The final warning/soft deadline must be before t_hard_s.")
        else:
            self.warning_times_s = tuple()
        if self.t_hard_s >= self.t_answer_s:
            raise ValueError("t_hard_s must be before t_answer_s.")

    @property
    def name(self):
        parts = [self.model_family]
        if self.use_priming:
            parts.append("primed")
        if self.use_hard_deadline:
            parts.append("hard")
        if self.use_soft_deadline:
            parts.append("soft")
        if self.use_calc_tool:
            parts.append("calc")
        return "+".join(parts)

### Model Loading

`load_model` loads the tokenizer and causal language model for the selected checkpoint. It uses 4-bit quantization and automatic device placement so the larger reasoning models can run within GPU memory constraints when CUDA is available.


In [24]:
def load_model(model_name):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Using device:", device)
    print("Loading model:", model_name)

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    quant_config = BitsAndBytesConfig(load_in_4bit=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=quant_config,
        device_map="auto",
    )
    return tokenizer, model, device

### Generation Stop Criteria

These custom stopping criteria let generation stop when either the time budget expires or a specific token sequence appears. The solver phases use them to enforce soft and hard deadlines during reasoning.


In [25]:
class StopOnTime(StoppingCriteria):
    def __init__(self, deadline):
        self.deadline = deadline

    def __call__(self, input_ids, scores, **kwargs):
        return time.time() >= self.deadline


class StopOnTokenSeq(StoppingCriteria):
    def __init__(self, token_ids):
        self.seq = token_ids

    def __call__(self, input_ids, scores, **kwargs):
        return len(self.seq) > 0 and input_ids[0, -len(self.seq) :].tolist() == self.seq

### Shared Model Runtime Wrapper

`ModelRuntime` wraps the tokenizer and model behind small convenience methods for appending text, decoding newly generated tokens, running timed generation, and building chat-template token inputs. This keeps the Qwen and DeepSeek solver code focused on strategy instead of low-level model calls.


In [26]:
class ModelRuntime:
    def __init__(self, model_name):
        self.tokenizer, self.model, self.device = load_model(model_name)

    def append_text(self, tokens, text):
        ids = self.tokenizer.encode(text, add_special_tokens=False)
        return torch.cat([tokens, torch.tensor([ids], device=tokens.device)], dim=-1)

    def decode_new(self, tokens, prev_len, skip_special_tokens=False):
        return self.tokenizer.decode(tokens[0, prev_len:], skip_special_tokens=skip_special_tokens)

    def generate(self, tokens, max_new, deadline, stop_sequences=None):
        criteria_list = [StopOnTime(deadline)]
        for seq in stop_sequences or []:
            criteria_list.append(StopOnTokenSeq(seq))

        mask = torch.ones((1, tokens.shape[-1]), device=tokens.device)
        with torch.no_grad():
            out = self.model.generate(
                tokens,
                attention_mask=mask,
                max_new_tokens=max_new,
                do_sample=False,
                temperature=None,
                top_p=None,
                pad_token_id=self.tokenizer.eos_token_id,
                stopping_criteria=StoppingCriteriaList(criteria_list),
            )
        return out[0].unsqueeze(0)

    def chat_tokens(self, messages):
        return self.tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            return_tensors="pt",
            return_dict=True,
        )["input_ids"].to(self.device)

### Qwen Calculator Tool

This block defines a restricted math namespace and a small tool-call executor for Qwen. When enabled, Qwen can emit a `math_tool` request, evaluate expressions with Python/SymPy utilities, and receive the result back in the conversation.


In [27]:
# =============================================================================
# Qwen-only math tool
# =============================================================================

x, y, z, n, t, a, b, c, k = symbols("x y z n t a b c k")

CALC_NAMESPACE = {
    "len": len,
    "list": list,
    "range": range,
    "sum": sum,
    "min": min,
    "max": max,
    "abs": abs,
    "round": round,
    "int": int,
    "float": float,
    "str": str,
    "enumerate": enumerate,
    "zip": zip,
    "map": map,
    "sorted": sorted,
    "reversed": reversed,
    "mean": mean,
    "median": median,
    "mode": mode,
    "stdev": stdev,
    "variance": variance,
    "x": x,
    "y": y,
    "z": z,
    "n": n,
    "t": t,
    "a": a,
    "b": b,
    "c": c,
    "k": k,
    "math": math,
    "comb": math.comb,
    "perm": math.perm,
    "ceil": math.ceil,
    "floor": math.floor,
    "log": math.log,
    "sqrt": math.sqrt,
    "sympy": sympy,
    "symbols": symbols,
    "solve": solve,
    "simplify": simplify,
    "expand": expand,
    "factor": factor,
    "integrate": integrate,
    "diff": diff,
    "limit": limit,
    "Matrix": Matrix,
    "Rational": Rational,
    "factorial": factorial,
    "binomial": binomial,
    "isprime": isprime,
    "factorint": factorint,
    "gcd": gcd,
    "lcm": lcm,
    "oo": oo,
    "E": E,
    "I": I,
    "pi": pi,
    "sin": sin,
    "cos": cos,
    "tan": tan,
    "exp": exp,
    "itertools": itertools,
    "linprog": linprog,
    "minimize": minimize,
    "minimize_scalar": minimize_scalar,
    "__builtins__": {},
}


def qwen_calc(expr):
    expr = expr.strip().rstrip(";")
    try:
        result = eval(expr, CALC_NAMESPACE)
        print(f"calc({expr!r}) -> {result}")
        return str(result)
    except NameError as exc:
        missing = re.findall(r"'(\w)'", str(exc))
        if missing:
            extra = {name: symbols(name) for name in missing}
            try:
                result = eval(expr, {**CALC_NAMESPACE, **extra})
                print(f"calc({expr!r}) -> {result} (auto-symbols: {missing})")
                return str(result)
            except Exception as exc2:
                return f"Error: {exc2}"
        return f"Error: {exc}"
    except Exception as exc:
        return f"Error: {exc}"

def execute_qwen_tool_call(raw_json):
    try:
        call = json.loads(raw_json.strip())
        if call.get("name") != "math_tool":
            return f"Unknown tool: {call.get('name')}"
        return qwen_calc(call.get("arguments", {}).get("expression", ""))
    except json.JSONDecodeError as exc:
        return f"JSON parse error: {exc}"
    except Exception as exc:
        return f"Tool execution error: {exc}"


QWEN_CALC_SYSTEM_PROMPT = """You are attending a quiz. Answer accurately in as few words as possible with the following format:

STEP A - In one sentence, state the idea to solve the problem.

STEP B - Solve the problem. For all calculations use the MATH TOOL described below for a variety of mathematical calculations.

STEP C - Look for the corresponding option and output the chosen option in ONE digit: 0, 1, 2, or 3. Nothing after it.

MATH TOOL (only for non-trivial arithmetic):
You have access to a math_tool that evaluates SymPy-compatible Python expressions.

To call it, use the following syntax:
<tool_call>
{"name": "math_tool", "arguments": {"expression": "YOUR_SYMPY_EXPRESSION"}}
</tool_call>

Don't forget to close the tool call tag after you opened it.

RULES:
- Use the smartest way to find the answer.
- Step 3 is the answer INDEX (0-3), not the computed value.
"""

QWEN_NO_TOOL_SYSTEM_PROMPT = """You are attending a quiz. Answer accurately in as few words as possible with the following format:

STEP A - In one sentence, state the idea to solve the problem.

STEP B - Solve the problem directly without calling tools.

STEP C - Look for the corresponding option and output the chosen option in ONE digit: 0, 1, 2, or 3. Nothing after it.

RULES:
- Use the smartest way to find the answer.
- Step 3 is the answer INDEX (0-3), not the computed value.
"""

### System Prompts

The prompt selector assigns the correct strategy instruction for each model family. DeepSeek is kept tool-free, while Qwen can run either with direct reasoning only or with the calculator-tool protocol.


In [28]:
DEEPSEEK_SYSTEM_PROMPT = (
    "You are a math expert. Think concisely and answer the multiple-choice "
    "question with a single digit: 0, 1, 2, or 3."
)


def system_prompt_for(cfg):
    if cfg.model_family == "deepseek":
        return DEEPSEEK_SYSTEM_PROMPT
    if cfg.use_calc_tool:
        return QWEN_CALC_SYSTEM_PROMPT
    return QWEN_NO_TOOL_SYSTEM_PROMPT

### Optional Priming Conversation

`build_priming_history` can warm up the model before the timed game starts. It sends the selected strategy as a short pre-game instruction and stores the model confirmation so later prompts can include the same conversational context.


In [29]:
def build_priming_history(runtime, cfg):
    if not cfg.use_priming:
        return []

    system_prompt = system_prompt_for(cfg)
    if cfg.model_family == "qwen":
        priming_messages = [
            {
                "role": "system",
                "content": "You are a quiz expert that answers multiple-choice questions.",
            },
            {
                "role": "user",
                "content": (
                    "Before we start, here is the strategy you must follow for every question.\n\n"
                    + system_prompt
                    + "\n\nConfirm you understood by summarizing the strategy in 3 bullet points."
                ),
            },
        ]
    else:
        priming_messages = [
            {"role": "system", "content": system_prompt},
            {
                "role": "user",
                "content": (
                    "Before we start, confirm the answering strategy in 3 short bullets. "
                    "Remember: the final output for each question must be only the option index."
                ),
            },
        ]

    tokens = runtime.chat_tokens(priming_messages)
    prev_len = tokens.shape[-1]
    tokens = runtime.generate(tokens, max_new=150, deadline=time.time() + 30.0)
    confirmation = runtime.decode_new(tokens, prev_len).replace("<|im_end|>", "").strip()
    print(f"Model confirmed strategy:\n{confirmation}\n")

    return [*priming_messages, {"role": "assistant", "content": confirmation}]

### Qwen Thinking Solver

The Qwen solver manages a timed reasoning phase, optional calculator-tool calls, speed-up prompts, wrap-up prompts, and final answer forcing. It is designed to let Qwen reason when time allows while still producing a valid option index before the deadline.


In [30]:
# =============================================================================
# Qwen solver
# =============================================================================

QWEN_SPEEDUP_MSG = "\nLet's reason a little bit faster:"
QWEN_WRAPUP_MSG = "\nI need to wrap up. In conclusion:"


def qwen_thinking_phase(runtime, tokens, t_start, cfg, options_text):
    tool_call_end_ids = runtime.tokenizer.encode("</tool_call>", add_special_tokens=False)
    im_end_ids = runtime.tokenizer.encode("<|im_end|>", add_special_tokens=False)

    gen_stops = [im_end_ids]
    if cfg.use_calc_tool:
        gen_stops.append(tool_call_end_ids)

    log = []
    tool_calls = 0
    t_hard = t_start + cfg.t_hard_s

    def force_answer(current_tokens, deadline, max_new, label):
        nudge = f"\nOptions: {options_text}. My final answer option index is "
        out = runtime.append_text(current_tokens, nudge)
        log.append(nudge)
        print(f"{label} nudge: '{nudge.strip()}'")

        prev_len = out.shape[-1]
        out = runtime.generate(out, max_new=max_new, deadline=deadline, stop_sequences=[])
        segment = runtime.decode_new(out, prev_len)
        log.append(segment)

        n_im = len(im_end_ids)
        if n_im > 0 and out.shape[-1] >= n_im and out[0, -n_im:].tolist() == im_end_ids:
            out = out[:, :-n_im]

        early_text = segment.replace("<|im_end|>", "").strip()
        print(f"✅ Answer: '{early_text}' ({time.time() - t_start:.1f}s).")
        return out, log, early_text

    def generate_until(deadline, max_new=150, wrapup=False):
        nonlocal tokens, tool_calls
        while True:
            prev_len = tokens.shape[-1]
            tokens = runtime.generate(
                tokens,
                max_new=max_new,
                deadline=deadline,
                stop_sequences=gen_stops,
            )
            segment = runtime.decode_new(tokens, prev_len)
            log.append(segment)

            elapsed = time.time() - t_start
            if wrapup:
                print(f"💭 wrap-up ({elapsed:.1f}s):\n{segment}\n")
            else:
                print(f"💭 ({elapsed:.1f}s):\n{segment}\n")

            natural_end = False
            n_im = len(im_end_ids)
            if n_im > 0 and tokens.shape[-1] >= n_im and tokens[0, -n_im:].tolist() == im_end_ids:
                tokens = tokens[:, :-n_im]
                natural_end = True

            if cfg.use_calc_tool:
                tool_matches = re.findall(r"<tool_call>(.*?)</tool_call>", segment, re.DOTALL)
                if tool_matches and tool_calls < cfg.max_tool_calls:
                    tool_calls += 1
                    result = execute_qwen_tool_call(tool_matches[-1])
                    tool_response = f"<tool_response>\n{result}\n</tool_response>\n"
                    tokens = runtime.append_text(tokens, tool_response)
                    log.append(tool_response)
                    continue

            already_answered = bool(
                re.search(r"\b[0-3]\b\s*(?:<\|im_end\|>)?\s*$", segment.strip())
            )
            if already_answered or natural_end:
                if wrapup:
                    print(f"✅ Wrapped up naturally ({elapsed:.1f}s).")
                else:
                    print(f"✅ Done ({elapsed:.1f}s).")
                return "done"

            return "deadline"

    if not cfg.use_hard_deadline:
        generate_until(t_start + cfg.t_answer_s)
        return tokens, log, None

    warning_deadlines = [t_start + t for t in cfg.warning_times_s]

    for idx, warning_deadline in enumerate(warning_deadlines):
        state = generate_until(warning_deadline)
        if state == "done":
            return tokens, log, None

        elapsed = time.time() - t_start
        is_last_warning = idx == len(warning_deadlines) - 1
        if is_last_warning:
            print(f"⏳ Wrap-up warning ({elapsed:.1f}s), nudging to conclude...")
            tokens = runtime.append_text(tokens, QWEN_WRAPUP_MSG)
            log.append(QWEN_WRAPUP_MSG)

            state = generate_until(t_hard, max_new=150, wrapup=True)
            if state == "done":
                return tokens, log, None

            print(f"🛑 Hard deadline ({time.time() - t_start:.1f}s), forcing answer.")
            return force_answer(
                tokens,
                deadline=t_start + cfg.t_answer_s,
                max_new=3,
                label="🛑 Hard",
            )

        print(
            f"⚡ Speed-up warning {idx}"
            f" ({elapsed:.1f}s), nudging to go faster..."
        )
        tokens = runtime.append_text(tokens, QWEN_SPEEDUP_MSG)
        log.append(QWEN_SPEEDUP_MSG)

    state = generate_until(t_hard)
    if state == "done":
        return tokens, log, None

    print(f"🛑 Hard deadline ({time.time() - t_start:.1f}s), forcing answer.")
    return force_answer(tokens, deadline=t_start + cfg.t_answer_s, max_new=3, label="🛑 Hard")


def build_qwen_solver(runtime, cfg, priming_history):
    def solver(question):
        options_text = " | ".join(f"{opt.id}={opt.text}" for opt in question.options)
        display_options = "\n".join(f"{opt.id}. {opt.text}" for opt in question.options)

        base_messages = priming_history or [{"role": "system", "content": system_prompt_for(cfg)}]
        messages = [
            *base_messages,
            {
                "role": "user",
                "content": f"Question: {question.text}\n\nOptions:\n{display_options}\n\n",
            },
        ]

        tokens = runtime.chat_tokens(messages)
        t_start = time.time()
        tokens, log, early = qwen_thinking_phase(runtime, tokens, t_start, cfg, options_text)

        if early is not None:
            answer_id = parse_final_answer(early, question)
            if answer_id is not None:
                print(f"[{cfg.name}] Option {answer_id} (nudged, {time.time() - t_start:.1f}s)")
                return answer_id, "\n".join(log)

        suffix = f"\nOptions: {options_text}. The answer option index is "
        print(f"🔚 Suffix: '{suffix.strip()}'")
        tokens = runtime.append_text(tokens, suffix)
        prev_len = tokens.shape[-1]
        tokens = runtime.generate(tokens, max_new=5, deadline=t_start + cfg.t_answer_s)
        response = runtime.decode_new(tokens, prev_len, skip_special_tokens=True).strip()

        print(f"🤖 [{cfg.name}] '{response}' (total: {time.time() - t_start:.1f}s)")
        answer_id = parse_final_answer(response, question)
        if answer_id is not None:
            print(f"✔️  Option {answer_id}")
            return answer_id, "\n".join(log)

        print("⚠️  Fallback to random")
        return random_solver(question)

    return solver

### DeepSeek Thinking Solver

The DeepSeek solver follows the same timed-answer idea but stays tool-free. It lets the model think until warning deadlines, nudges it to speed up or conclude when necessary, and then extracts a final multiple-choice option.


In [31]:
# =============================================================================
# DeepSeek solver, intentionally tool-free
# =============================================================================

DEEPSEEK_SPEEDUP_MSG = "\nLet's reason a little bit faster:"
DEEPSEEK_WRAPUP_MSG = "\nI need to wrap up. In conclusion:"


def deepseek_thinking_phase(runtime, tokens, warning_deadlines, t_hard, t_start):
    think_end = runtime.tokenizer.encode("</think>", add_special_tokens=False)
    log = []
    warnings = list(warning_deadlines)

    while warnings:
        current_deadline = warnings.pop(0)
        is_last = len(warnings) == 0

        while True:
            prev_len = tokens.shape[-1]
            tokens = runtime.generate(
                tokens,
                max_new=4096,
                deadline=current_deadline,
                stop_sequences=[think_end],
            )
            segment = runtime.decode_new(tokens, prev_len)
            log.append(segment)

            elapsed = time.time() - t_start
            print(f"💭 ({elapsed:.1f}s):\n{segment}\n")

            if "</think>" in segment:
                print(f"✅ Finished naturally in {elapsed:.1f}s.")
                return tokens, log

            break

        if is_last:
            print(f"⏳ Wrap-up warning ({elapsed:.1f}s), nudging to conclude...")
            tokens = runtime.append_text(tokens, DEEPSEEK_WRAPUP_MSG)
            log.append(DEEPSEEK_WRAPUP_MSG)

            prev_len = tokens.shape[-1]
            tokens = runtime.generate(
                tokens,
                max_new=4096,
                deadline=t_hard,
                stop_sequences=[think_end],
            )
            segment = runtime.decode_new(tokens, prev_len)
            log.append(segment)

            elapsed = time.time() - t_start
            print(f"💭 wrap-up ({elapsed:.1f}s):\n{segment}\n")

            if "</think>" in segment:
                print(f"✅ Wrapped up naturally ({elapsed:.1f}s).")
            else:
                print(f"🛑 Hard deadline ({elapsed:.1f}s), forcing </think>.")
                tokens = runtime.append_text(tokens, "</think>\n")
                log.append("</think>  [FORCED]")
        else:
            print(
                f"⚡ Speed-up warning {len(warning_deadlines) - len(warnings) - 1}"
                f" ({elapsed:.1f}s), nudging to go faster..."
            )
            tokens = runtime.append_text(tokens, DEEPSEEK_SPEEDUP_MSG)
            log.append(DEEPSEEK_SPEEDUP_MSG)

    return tokens, log


def build_deepseek_solver(runtime, cfg, priming_history):
    def solver(question):
        options_text = "\n".join(f"{opt.id}. {opt.text}" for opt in question.options)
        base_messages = priming_history or [{"role": "system", "content": system_prompt_for(cfg)}]
        messages = [
            *base_messages,
            {
                "role": "user",
                "content": f"Question: {question.text}\n\nOptions:\n{options_text}\n\nAnswer:",
            },
        ]

        tokens = runtime.chat_tokens(messages)
        tokens = runtime.append_text(tokens, "<think>\nLet's start:\n")

        t_start = time.time()
        warning_deadlines = [t_start + t for t in cfg.warning_times_s]

        if cfg.use_hard_deadline:
            if warning_deadlines:
                tokens, log = deepseek_thinking_phase(
                    runtime,
                    tokens,
                    warning_deadlines,
                    t_start + cfg.t_hard_s,
                    t_start,
                )
            else:
                think_end = runtime.tokenizer.encode("</think>", add_special_tokens=False)
                prev_len = tokens.shape[-1]
                tokens = runtime.generate(
                    tokens,
                    max_new=4096,
                    deadline=t_start + cfg.t_hard_s,
                    stop_sequences=[think_end],
                )
                log = [runtime.decode_new(tokens, prev_len)]
                if "</think>" not in log[-1]:
                    tokens = runtime.append_text(tokens, "</think>\n")
                    log.append("</think> [FORCED]")
        else:
            log = []

        tokens = runtime.append_text(tokens, "\nThe answer index is ")
        prev_len = tokens.shape[-1]
        tokens = runtime.generate(tokens, max_new=10, deadline=t_start + cfg.t_answer_s)
        response = runtime.decode_new(tokens, prev_len, skip_special_tokens=True).strip()

        print(f"🤖 '{response}' (total: {time.time() - t_start:.1f}s)")
        answer_id = parse_final_answer(response, question)
        if answer_id is not None:
            print(f"✔️  Option {answer_id}")
            return answer_id, "\n".join(log)

        print("⚠️  Fallback to random")
        return random_solver(question)

    return solver

### Solver Selection

`build_solver` chooses the correct solver builder from the configuration: Qwen configurations use the Qwen solver, DeepSeek configurations use the DeepSeek solver, and unsupported model families fail early.


In [32]:
def build_solver(runtime, cfg, priming_history):
    if cfg.model_family == "qwen":
        return build_qwen_solver(runtime, cfg, priming_history)
    if cfg.model_family == "deepseek":
        return build_deepseek_solver(runtime, cfg, priming_history)
    raise ValueError(f"Unsupported model family: {cfg.model_family}")


### Result Summary Metrics

`summarize_results` aggregates per-game logs into comparison metrics such as mean level reached, decision-time variability, total timeouts, and best level reached. These metrics make the ablations directly comparable across models and strategy variants.


In [33]:
def summarize_results(game_df):
    metrics = [
        "levels_reached",
        "decision_time",
        "timeouts",
    ]

    grouped = game_df.groupby(["solver", "model"], dropna=False)
    rows = []
    for (solver, model), group in grouped:
        row = {
            "solver": solver,
            "model": model,
            "games_played": len(group),
            "timeouts_total": pd.to_numeric(group["timeouts"], errors="coerce").sum(),
        }
        for metric in metrics:
            values = pd.to_numeric(group[metric], errors="coerce")
            row[f"{metric}_mean"] = values.mean()
            row[f"{metric}_std"] = values.std(ddof=1)

        row['level_reached_max'] = pd.to_numeric(group['levels_reached'], errors="coerce").max()
        rows.append(row)


    return pd.DataFrame(rows)

### Ablation Runner

`run_ablation` executes each experiment configuration across multiple games. It reuses an already-loaded model when consecutive configurations use the same checkpoint, clears GPU memory when switching models, runs optional priming, plays each game automatically, then writes per-game results, per-question logs, and summary CSV files.


In [34]:
def run_ablation(client, competition_id, ablations, output_dir):
    competitions = client.competitions.list_all()
    print("\n=== Available Competitions ===")
    for comp in competitions:
        print(f"{comp.id}: {comp.name} ({comp.max_levels} questions)")

    current_model_name = None
    current_runtime = None

    for cfg in ablations:
        print(f"\n{'=' * 60}\nABLATION: {cfg.name}\nMODEL: {cfg.model_name}\n{'=' * 60}")

        if cfg.model_name != current_model_name:
            if current_runtime is not None:
                del current_runtime
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
            current_runtime = ModelRuntime(cfg.model_name)
            current_model_name = cfg.model_name
        runtime = current_runtime

        priming_history = build_priming_history(runtime, cfg)
        solver = build_solver(runtime, cfg, priming_history)

        for game_idx in range(cfg.num_games):
            run_id = f"{cfg.name}__game_{game_idx + 1:03d}"
            print(f"\nGame {game_idx + 1}/{cfg.num_games}: {run_id}")
            game = client.game.start(competition_id=competition_id)
            play_game_auto(
                game,
                solver,
                solver_name=cfg.name,
                model_name=cfg.model_name,
                run_id=run_id,
            )

    game_df = pd.DataFrame(game_logs)
    question_df = pd.DataFrame(run_logs)
    summary_df = summarize_results(game_df)

    print("\n=== Per-game results ===")
    print(game_df.to_string(index=False))

    print("\n=== Mean and standard-deviation comparison ===")
    print(summary_df.to_string(index=False))

    os.makedirs(output_dir, exist_ok=True)
    game_df.to_csv(os.path.join(output_dir, "polimillionaire_game_results.csv"), index=False)
    question_df.to_csv(os.path.join(output_dir, "polimillionaire_question_logs.csv"), index=False)
    summary_df.to_csv(os.path.join(output_dir, "polimillionaire_summary_stats.csv"), index=False)

    return game_df, question_df, summary_df

### DeepSeek Ablation Setup

These DeepSeek runs isolate the impact of the soft-deadline wrap-up behavior. Both variants use the same model and no calculator tool; the comparison is whether a timed warning-and-wrap-up prompt improves final answer reliability before the hard deadline.


In [35]:
ablations = [
    ExperimentConfig(
        model_name=DEEPSEEK_MODEL,
        model_family="deepseek",
        use_priming=False,
        use_hard_deadline=True,
        use_soft_deadline=True,
        use_calc_tool=False,
        t_soft_s=22.0,
        t_hard_s=26.5,
        t_answer_s=29.0,
        num_games=15,
    ),

    ExperimentConfig(
        model_name=DEEPSEEK_MODEL,
        model_family="deepseek",
        use_priming=False,
        use_hard_deadline=True,
        use_soft_deadline=False,
        use_calc_tool=False,
        t_soft_s=22.0,
        t_hard_s=26.5,
        t_answer_s=29.0,
        num_games=15,
    )
]

### Run DeepSeek Experiments

This cell launches the DeepSeek math experiments and writes the resulting CSV files into `deepseek_results`. It can take a while because it plays multiple full games with the local model.


In [36]:
game_df, question_df, summary_df = run_ablation(client, comp_id, ablations, 'deepseek_results')


=== Available Competitions ===
0: Entertainment (15 questions)
1: Ancient History and Politics (15 questions)
2: Science and Nature (15 questions)
3: Maths (15 questions)
4: Philosophy and Psychology (15 questions)
5: News (15 questions)

ABLATION: deepseek+hard+soft
MODEL: deepseek-ai/DeepSeek-R1-Distill-Qwen-7B
Using device: cuda
Loading model: deepseek-ai/DeepSeek-R1-Distill-Qwen-7B


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-000002.safetensors:   0%|          | 0.00/8.61G [00:00<?, ?B/s]

model-00002-of-000002.safetensors:   0%|          | 0.00/6.62G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]


Game 1/15: deepseek+hard+soft__game_001

--- Level 1 ---
Question:
          (i + 1)(5 – 5i)(5 + 5i) =

          Options:
          0. 50 + 50i
1. 25 – 25i
2. 50 – 50i
3. 25 + 25i

          Answer:

Time remaining: 29.9s


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


💭 (11.3s):
First, I'll expand the expression (i + 1)(5 – 5i)(5 + 5i).

I notice that (5 – 5i)(5 + 5i) is a difference of squares, so it simplifies to 25 + 25i².

Since i² = -1, this becomes 25 - 25 = 0.

Now, the expression simplifies to (i + 1) * 0 = 0.

So, the final answer is 0.
</think>

✅ Finished naturally in 11.3s.
🤖 '0.

**Step-by-Step Explanation:**' (total: 13.2s)
✔️  Option 0

Selected answer: 0
Decision time: 13.31s
CORRECT!

--- Level 2 ---
Question:
          Let $f(x)=3x+4$ and $g(x)=2x-3$. If $h(x)=f(g(x))$, then what is the inverse of $h(x)$?

          Options:
          0. \frac{x+5}{6}
1. \frac{x+5}{3}
2. \frac{x-5}{6}
3. \frac{x-5}{3}

          Answer:

Time remaining: 29.9s
💭 (22.1s):
First, I need to find the composition of functions h(x) = f(g(x)). 

So, I'll substitute g(x) into f(x). That means wherever I see an x in f(x), I'll replace it with g(x). 

Given f(x) = 3x + 4 and g(x) = 2x - 3, substituting g(x) into f(x) gives:
h(x) = 3*(2x - 3) + 4.

Next, I'll 

### Inspect DeepSeek Summary

As we can see in the following table, using the soft deadline leads to better results, both in terms of mean level reached and max level reached. Moreover, the decision time is also faster in this setup. However, the standard deviation is a little bit higher than the hard setup, indicating more variance in the performance of the model.

In [37]:
summary_df.head()

,solver,model,games_played,timeouts_total,levels_reached_mean,levels_reached_std,decision_time_mean,decision_time_std,timeouts_mean,timeouts_std,level_reached_max
0,deepseek+hard,deepseek-ai/DeepSeek-R1-Distill-Qwen-7B,15,0,2.666667,1.799471,23.932217,3.721685,0.0,0.0,7
1,deepseek+hard+soft,deepseek-ai/DeepSeek-R1-Distill-Qwen-7B,15,0,3.266667,2.250926,23.565408,3.608467,0.0,0.0,10


### Qwen Ablation Setup

These Qwen runs isolate the impact of the calculator tool. Both variants use priming and the same deadline schedule; the comparison is whether tool-assisted symbolic/numeric calculation improves math-game performance over Qwen's direct reasoning alone.


In [38]:
ablations = [
    ExperimentConfig(
        model_name=QWEN_MODEL,
        model_family="qwen",
        use_priming=True,
        use_hard_deadline=True,
        use_soft_deadline=True,
        warning_times_s=(22.0,),
        use_calc_tool=False,
        t_soft_s=22.0,
        t_hard_s=26.5,
        t_answer_s=29.0,
        num_games=15,
    ),
    ExperimentConfig(
        model_name=QWEN_MODEL,
        model_family="qwen",
        use_priming=True,
        use_hard_deadline=True,
        use_soft_deadline=True,
        warning_times_s=(22.0,),
        use_calc_tool=True,
        t_soft_s=22.0,
        t_hard_s=26.5,
        t_answer_s=29.0,
        num_games=15,
    ),
]

### Run Qwen Experiments

This cell launches the Qwen math experiments and writes the resulting CSV files into `qwen_results`. It reuses the same ablation runner so the Qwen and DeepSeek summaries are directly comparable.


In [39]:
game_df, question_df, summary_df = run_ablation(client, comp_id, ablations, 'qwen_results')


=== Available Competitions ===
0: Entertainment (15 questions)
1: Ancient History and Politics (15 questions)
2: Science and Nature (15 questions)
3: Maths (15 questions)
4: Philosophy and Psychology (15 questions)
5: News (15 questions)

ABLATION: qwen+primed+hard+soft
MODEL: Qwen/Qwen2.5-7B-Instruct
Using device: cuda
Loading model: Qwen/Qwen2.5-7B-Instruct


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model-00004-of-00004.safetensors:   0%|          | 0.00/3.56G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:650: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Model confirmed strategy:
- Answer each question concisely in one sentence.
- Provide the solution directly without using external tools.
- Choose the answer index (0-3) as the final output.


Game 1/15: qwen+primed+hard+soft__game_001

--- Level 1 ---
Question:
          A university’s mathematics department has 10 professors and will offer 20 different courses next semester. Each professor will be assigned to teach exactly 2 of the courses, and each course will have exactly one professor assigned to teach it. If any professor can be assigned to teach any course, how many different complete assignments of the 10 professors to the 20 courses are possible?

          Options:
          0. 10^(20) - 2^(10)
1. 20!/2^(10)
2. 10!/2^9
3. 10^(20) - 100

          Answer:

Time remaining: 29.9s
💭 (18.1s):
STEP A - Calculate the number of ways to assign 10 professors to 20 courses such that each professor teaches exactly 2 courses and each course has exactly one professor.
STEP B - First, choos

### Inspect Final Summary

As you can see in the following table, Qwen model with calculator tool performs slightly better than the Deepseek model with soft deadline. However, the Qwen setup has also hit the timeout 3 times, indicating that it might get stuck in certain questions. It also performs with a large variance.

As a result, we consider Deepseek with soft ddeadline as our best model.

In [40]:
summary_df.head()

,solver,model,games_played,timeouts_total,levels_reached_mean,levels_reached_std,decision_time_mean,decision_time_std,timeouts_mean,timeouts_std,level_reached_max
0,deepseek+hard,deepseek-ai/DeepSeek-R1-Distill-Qwen-7B,15,0,2.666667,1.799471,23.932217,3.721685,0.0,0.000000,7
1,deepseek+hard+soft,deepseek-ai/DeepSeek-R1-Distill-Qwen-7B,15,0,3.266667,2.250926,23.565408,3.608467,0.0,0.000000,10
2,qwen+primed+hard+soft,Qwen/Qwen2.5-7B-Instruct,15,0,2.333333,1.676163,13.246093,4.738451,0.0,0.000000,7
3,qwen+primed+hard+soft+calc,Qwen/Qwen2.5-7B-Instruct,15,3,3.333333,2.350279,18.714999,5.995511,0.2,0.414039,10
